In [ ]:
# %pip install google-adk
# %pip install googlemaps
# %pip install litellm

In [ ]:
import requests
from typing import Dict, Any, Optional

def GetWeatherForecast(lat: float, lon: float) -> Optional[Dict[str, Any]]:
    """
    Retrieves current weather data from the National Weather Service API
    based on latitude and longitude.

    Args:
        lat: The latitude of the location (e.g., 38.8951).
        lon: The longitude of the location (e.g., -77.0364).

    Returns:
        A dictionary containing current weather data, or None if the request fails.
        Raises an error if the coordinates are outside the US.

    Example:
        >>> weather = get_nws_weather(38.8951, -77.0364)
        >>> if weather:
        ...     print(f"Temperature: {weather['temperature']}°C")
    """
    # 1. Get the grid endpoint (points to NWS office and grid square)
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    headers = {"User-Agent": "(myweatherapp.com, contact@email.com)"}

    try:
        response = requests.get(points_url, headers=headers)
        response.raise_for_status()
        point_data = response.json()

        # 2. Extract forecast URL from points data
        forecast_url = point_data["properties"]["forecast"]

        # 3. Get the actual forecast data
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()

        # Return the current period's forecast
        return forecast_data["properties"]["periods"][0]

    except requests.exceptions.RequestException as e:
        print(f"Error fetching weather data: {e}")
        return None


In [ ]:
import os
from typing import Optional, Tuple
import googlemaps

def GetLongitudeLatitude(place_name: str) -> str:
    """
    Converts a place name or address into geographic coordinates (latitude and longitude)
    using the Google Maps Geocoding API.

    Args:
        place_name: The address or place name to geocode.
        api_key: Your Google Maps API key.

    Returns:
        A str containing the (latitude, longitude) as floats if successful,
        otherwise None.
    """
    gmaps = googlemaps.Client(key="API_KEY")
    try:
        # Geocoding the address
        geocode_result = gmaps.geocode(place_name)

        if geocode_result:
            # Extracting latitude and longitude from the first result
            location = geocode_result[0]['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            return 'latitude :'+str(latitude)+" & longitude: "+ str(longitude)
        else:
            print(f"No geocoding results found for '{place_name}'.")
            return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


In [ ]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmResponse, LlmRequest
import logging

# Configure basic logging at the start of your script
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s'
)

# Get a logger instance for your specific module/agent
logger = logging.getLogger(__name__)

def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest)-> Optional[LlmResponse]:
  if llm_request.contents:
    last = llm_request.contents[-1]
    if last.role == "user" and last.parts and last.parts[0].text:
      logger.info("[%s] USER » %s", callback_context.agent_name,last.parts[0].text)
  return None

In [ ]:
def log_model_response(
callback_context: CallbackContext, llm_response: LlmResponse)-> Optional[LlmResponse]:
  if llm_response.content and llm_response.content.parts:
    txt = llm_response.content.parts[0].text
    if txt:
      logger.info("[%s] MODEL » %s", callback_context.agent_name, txt.strip())
  return None

In [ ]:
# 2. Define a pre-processing hook to check input
def check_user_input(user_message: str):
    # Simple validation logic
    if "forbidden_word" in user_message.lower():
        print("Input blocked due to restricted content.")
        return "BAD"
    return "GOOD"


In [ ]:
def moderate_user_prompt(callback_context: CallbackContext,llm_request: LlmRequest) -> Optional[LlmResponse]:
  try:
    if not llm_request.contents:
      return None
    last = llm_request.contents[-1]
    user_text = last.parts[0].text.strip()
    result_text = check_user_input(user_text)
    if result_text.strip().upper() == "BAD":
      return LlmResponse(content={
        "role": "model",
        "parts": [{"text": "Message violates our content guidelines."}]})
  except Exception as e:
    import logging
    logging.exception("Moderation callback failed: %s", e)
  return None

In [ ]:
def chained_before_callback(callback_context, llm_request):
  # 1. Moderation check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result # STOP: message was inappropriate
  # 2. Log user input (optional)
  log_user_prompt(callback_context, llm_request)
  return None # Allow agent to proceed

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

weather_agent = Agent(
  name="weather_agent",
  model=LiteLlm(model="gemini-2.5-flash"),
  instruction=(
  """You are a Weather Agent. Provide accurate forecasts using only these tools when needed:
   GetLongitudeLatitude(location_string) → {lat, lon} and
   GetWeatherForecast(lat, lon) → forecast data. If the user gives coordinates,
   call GetWeatherForecast directly; otherwise call GetLongitudeLatitude then GetWeatherForecast.
   """
  ),
  tools=[GetWeatherForecast, GetLongitudeLatitude],
  before_model_callback=chained_before_callback,
  after_model_callback=log_model_response,
)

In [ ]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
  agent=weather_agent,
  enable_tracing=False,
)

In [ ]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
print(session.id)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


c54b1600-b266-49c9-bbff-dee714a42a4c


In [ ]:
from IPython.display import Markdown, display
for event in app.stream_query(
user_id=user_id,
session_id=session.id,
message="what is the weather of new york",
):
  lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

ERROR:root:Moderation callback failed: 'NoneType' object has no attribute 'strip'
Traceback (most recent call last):
  File "/tmp/ipython-input-2169104461.py", line 6, in moderate_user_prompt
    user_text = last.parts[0].text.strip()
                ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'strip'
ERROR:root:Moderation callback failed: 'NoneType' object has no attribute 'strip'
Traceback (most recent call last):
  File "/tmp/ipython-input-2169104461.py", line 6, in moderate_user_prompt
    user_text = last.parts[0].text.strip()
                ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'strip'
Exception in thread Thread-56 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/vertex_ai/gemini/vertex_and_google_ai_studio_gemini.py", line 2606, in async_completion
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/p


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



KeyError: 'text'